In [ ]:
import random
#Cards implementation
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value
    def __repr__(self):
        return f"{self.color} {self.value}"
def Deck():
    colors = ['Red', 'Yellow', 'Blue', 'Green']
    values = [str(i) for i in range(10)] + ['Skip']
    deck = [Card(c, v) for c in colors for v in values]
    random.shuffle(deck)
    return deck
def initialise():
    deck = Deck()
    states = {
        # 5 cards for each player
        'p1' : [deck.pop() for _ in range(5)],
        'p2' : [deck.pop() for _ in range(5)],
        'p3' : [deck.pop() for _ in range(5)],
        # initial card to play with
        'top_card' : deck.pop(),
        'deck' : deck
    }
    return states

def get_valid_moves(hand, top_card):
    valid_moves = []
    for card in hand:
        # checks if same color or number for the valid move
        if (card.color == top_card.color or card.value == top_card.value):
            valid_moves.append(card)
    return valid_moves

def apply_move(states, move, player_key):
    new_state = {
        # copying cards 
        'p1' : states['p1'][:],
        'p2' : states['p2'][:],
        'p3' : states['p3'][:], 
        'top_card' : states['top_card'],
        'deck' : states['deck'][:]
    }
    # if player doesnt have the card, he/she draws one from deck
    if move is None:
        if new_state['deck']:
            drawn = new_state['deck'].pop()
            new_state[player_key].append(drawn)
        return new_state, False
    # if player has the card and is playing it 
    new_state[player_key].remove(move)
    # updates the top_card to play against
    new_state['top_card'] = move
    # checks if the played card is a skip
    is_skip = (move.value == 'Skip')
    return new_state, is_skip

# Phase 2 - Making the Evalution Fucntion 
def evaluate(state, player_key, strategy):
    # Determine who the opponents are
    opponents = [p for p in ['p1', 'p2', 'p3'] if p != player_key]
    
    c_ai = len(state[player_key])  # Cards in AI's hand
    # Average cards held by opponents
    c_opp = (len(state[opponents[0]]) + len(state[opponents[1]])) / 2.0
    # Number of Skip cards in hand 
    s = sum(1 for card in state[player_key] if card.value == 'Skip')
    
    if strategy == "defensive":
        # Value Skips higher and worry more about opponents finishing 
        return 50 - (5 * c_ai) + (4 * c_opp) + (6 * s)
    else:
        # Prioritize getting rid of your own cards quickly
        return 50 - (10 * c_ai) + (2 * c_opp) + (2 * s)
        
# Phase 3 - Implementing the expectimax and minimax algorithms for the players
def minimax(state, depth, is_maximizing, player_turn, ai_player):
    # Check for terminal state (win) or depth limit reached 
    # Check if any player has 0 cards 
    if depth == 0 or len(state['p1']) == 0 or len(state['p2']) == 0 or len(state['p3']) == 0:
        return evaluate(state, ai_player, "defensive") 

    players = ['p1', 'p2', 'p3']
    current_player = players[player_turn]
    # fetches the valid moves for every player
    valid_moves = get_valid_moves(state[current_player], state['top_card']) 

    # MAXIMIZING PLAYER 
    if is_maximizing:
        best_score = float('-inf')
        
        # If no moves, must draw a card from the deck
        if not valid_moves:
            # calls apply_move functions drawing card functionality
            new_state, _ = apply_move(state, None, current_player)
            next_turn = (player_turn + 1) % 3
            # After drawing, it might be the AI's turn again (if everyone else is skipped)
            next_is_max = (players[next_turn] == ai_player)
            return minimax(new_state, depth - 1, next_is_max, next_turn, ai_player)
        
        for move_card in valid_moves:
            new_state, skip = apply_move(state, move_card, current_player) 
            # Skip Logic: Next player's turn is skipped 
            next_turn = (player_turn + 2 if skip else player_turn + 1) % 3
            next_is_max = (players[next_turn] == ai_player)
            
            score = minimax(new_state, depth - 1, next_is_max, next_turn, ai_player)
            best_score = max(best_score, score)
        return best_score

    # MINIMIZING PLAYERS 
    else:
        best_score = float('inf')
        
        # If opponent has no moves, they draw card from the deck
        if not valid_moves:
            new_state, _ = apply_move(state, None, current_player)
            next_turn = (player_turn + 1) % 3
            next_is_max = (players[next_turn] == ai_player)
            return minimax(new_state, depth - 1, next_is_max, next_turn, ai_player)
        
        for move_card in valid_moves:
            new_state, skip = apply_move(state, move_card, current_player) 
            # Handle turn transition
            next_turn = (player_turn + 2 if skip else player_turn + 1) % 3
            next_is_max = (players[next_turn] == ai_player)
            
            # The opponent chooses the move that MINIMIZES the AI's evaluation 
            score = minimax(new_state, depth - 1, next_is_max, next_turn, ai_player)
            best_score = min(best_score, score)
        return best_score
        
def expectimax(state, depth, player_turn, ai_player):
    # Checking if any plaer is outta cards (winning condition)
    if depth == 0 or len(state['p1']) == 0 or len(state['p2']) == 0 or len(state['p3']) == 0:
        return evaluate(state, ai_player, "offensive") 

    players = ['p1', 'p2', 'p3']
    current_player = players[player_turn]
    valid_moves = get_valid_moves(state[current_player], state['top_card'])

    # AI's Turn (MAX Node) 
    if current_player == ai_player:
        if not valid_moves:
            # Must Draw
            return chance_node_value(state, depth, player_turn, ai_player)
        
        # Maximize the outcome of available moves
        scores = []
        for m in valid_moves:
            new_state, skip = apply_move(state, m, current_player) 
            next_turn = (player_turn + 2 if skip else player_turn + 1) % 3 
            scores.append(expectimax(new_state, depth - 1, next_turn, ai_player))
        return max(scores)

    # Opponent's Turn (Chance Node) 
    else:
        if not valid_moves:
            # Opponent draws a card
            new_state, _ = apply_move(state, None, current_player) 
            next_turn = (player_turn + 1) % 3
            return expectimax(new_state, depth - 1, next_turn, ai_player)
        
        # Expectimax assumes opponents play a random legal move
        # So we take the Average of all legal moves 
        total_score = 0
        for m in valid_moves:
            new_state, skip = apply_move(state, m, current_player)
            next_turn = (player_turn + 2 if skip else player_turn + 1) % 3 
            total_score += expectimax(new_state, depth - 1, next_turn, ai_player)
        
        return total_score / len(valid_moves)

def chance_node_value(state, depth, player_turn, ai_player):
    # Calculates the expected value of drawing from the current deck
    if not state['deck']:
        return evaluate(state, ai_player, "offensive")
    new_state, _ = apply_move(state, None, ['p1', 'p2', 'p3'][player_turn]) 
    next_turn = (player_turn + 1) % 3
    return expectimax(new_state, depth - 1, next_turn, ai_player)
def test_three_player_simulation(state):
    # Mapping players to their assigned strategies 
    player_strategies = {
        'p1': 'defensive',
        'p2': 'offensive',
        'p3': 'defensive' 
    }
    
    print(f"=== INITIAL STATE ===")
    print(f"Top card: {state['top_card']}\n")

    for player_key in ['p1', 'p2', 'p3']:
        strategy = player_strategies[player_key]
        hand = state[player_key]
        
        print(f"--- {player_key.upper()} ({strategy.capitalize()}) Turn ---")
        print(f"Hand: {hand}")
        
        #  Gets and creates a list of all legal moves 
        valid_moves = get_valid_moves(hand, state['top_card'])
        
        print(f"AI decision (All possible decisions considered at depth 1):")
        
        #  Draw card from deck if not valid move 
        if not valid_moves:
            temp_state, _ = apply_move(state, None, player_key)
            score = evaluate(temp_state, player_key, strategy)
            print(f"Action: Draw Card")
            print(f"Expected score: {score}")
        
        # Evaluate each legal play (calcultating score for each move the player may make)
        else:
            for m in valid_moves:
                # Simulate the move to see the resulting state 
                temp_state, _ = apply_move(state, m, player_key)
                # Calculate the evaluation score for this path
                score = evaluate(temp_state, player_key, strategy)
                print(f"Play: {m}")
                print(f"Expected score: {score}")
        
        print("-" * 40)